# Reproducible experiment: K-Means vs MiniBatchKMeans

**Purpose:** evaluate how training-set size affects clustering quality and fit time on UCI Online Retail transaction rows.

## Final protocol (version 1.0)

- Remove cancelled invoices (`InvoiceNo` starts with `C`), and rows with `Quantity <= 0` or `UnitPrice <= 0`.
- Retain missing `CustomerID` and exact duplicate rows. `CustomerID` is not a clustering feature.
- Prevent invoice-level leakage: split by `InvoiceNo` groups into a training pool (80%) and a fixed holdout pool (20%).
- For each of 30 seeds, create **nested training subsets** of 1,000, 5,000, 10,000, 25,000, 50,000 and 100,000 rows from the training pool.
- Evaluate all fitted models on the **same fixed evaluation sample** drawn from the holdout pool.
- Compare two feature scenarios: (A) `Quantity`, `UnitPrice`; (B) those two plus `TotalAmount = Quantity × UnitPrice`.
- Apply `log1p`, then fit `StandardScaler` on each training subset and transform its matching fixed evaluation set.
- Compare KMeans and MiniBatchKMeans with `k=5`, `k-means++`, `max_iter=300`, `tol=1e-4`, `n_init=1` and `n_init=10`. For each n_init, both algorithms receive the same seed.
- Compute evaluation Silhouette, Davies–Bouldin, Calinski–Harabasz, evaluation inertia per observation, and ARI between the two algorithms' evaluation partitions.
- Record fit time using repeated timed fits; report median and all repetitions.

**Important:** this notebook generates a new experiment. Its results must replace—not be presented as—the unverified numbers in the earlier manuscript. The holdout/nested-sampling design differs from the old independently sampled design. Update the Methods, Results, Discussion, Abstract and Conclusion accordingly. This notebook does not claim to reproduce the earlier reported experiment.


## 1. Install dependencies and import libraries

In [ ]:
!pip -q install scikit-learn==1.8.0 openpyxl psutil

In [ ]:
import os, sys, time, json, platform, warnings, gc, hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score
)
from threadpoolctl import threadpool_limits, threadpool_info

SEED_LIST = list(range(42, 72))  # 30 seeds
SAMPLE_SIZES = [1_000, 5_000, 10_000, 25_000, 50_000, 100_000]
N_CLUSTERS = 5
N_INIT_VALUES = [1, 10]
BATCH_SIZE = 256
MAX_ITER = 300
TOL = 1e-4
EVAL_SIZE = 10_000       # fixed evaluation sample size
SILHOUETTE_SIZE = 5_000  # fixed subsample for silhouette computation
TIME_REPEATS = 3
THREAD_LIMIT = 1
HOLDOUT_FRACTION = 0.20
HOLDOUT_SEED = 20261109
OUTPUT_DIR = "/content/experiment_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Thread pools before limiting:", threadpool_info())


## 2. Upload the dataset

Upload the original `Online Retail.xlsx` file when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()

xlsx_files = [name for name in uploaded if name.lower().endswith((".xlsx", ".xls"))]
if not xlsx_files:
    raise FileNotFoundError("Upload Online Retail.xlsx (Excel format).")
DATA_PATH = xlsx_files[0]
print("Using:", DATA_PATH)


## 3. Load and clean the data

This cleaning follows the article's stated filters. Missing `CustomerID` and duplicate rows are deliberately retained. The notebook records the row counts so the Methods section can report them accurately.

In [ ]:
raw = pd.read_excel(DATA_PATH, engine="openpyxl")
required = {"InvoiceNo", "Quantity", "UnitPrice"}
missing_cols = required - set(raw.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

n_raw = len(raw)
invoice_text = raw["InvoiceNo"].astype(str).str.strip()
not_cancelled = ~invoice_text.str.upper().str.startswith("C")
df = raw.loc[not_cancelled].copy()
n_after_cancel = len(df)

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df = df.dropna(subset=["Quantity", "UnitPrice"])
df = df.loc[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
n_after_positive = len(df)

df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["Quantity", "UnitPrice", "TotalAmount"]).copy()

cleaning_summary = pd.DataFrame([
    ["Original rows", n_raw],
    ["After removing cancelled invoices", n_after_cancel],
    ["After removing non-positive/missing Quantity or UnitPrice", n_after_positive],
    ["Final eligible rows", len(df)],
    ["Missing CustomerID retained", int(df["CustomerID"].isna().sum()) if "CustomerID" in df.columns else None],
    ["Exact duplicate rows retained", int(df.duplicated().sum())],
], columns=["Item", "Count"])
display(cleaning_summary)
cleaning_summary.to_csv(f"{OUTPUT_DIR}/cleaning_summary.csv", index=False)

if len(df) < max(SAMPLE_SIZES) + EVAL_SIZE:
    raise ValueError("Not enough eligible rows for the selected training and evaluation sizes.")


## 4. Create a fixed holdout split by invoice

Rows belonging to the same invoice are assigned to the same side of the split. This reduces leakage between training and evaluation. The split is fixed and independent of the 30 experimental seeds.

The evaluation sample is drawn once from the holdout pool and reused for every model. The training subsets are nested within the training pool for each seed.

In [ ]:
# Group-aware 80/20 split using InvoiceNo as group.
groups = df["InvoiceNo"].astype(str).to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=HOLDOUT_FRACTION, random_state=HOLDOUT_SEED)
train_idx, holdout_idx = next(gss.split(df, groups=groups))

train_pool = df.iloc[train_idx].reset_index(drop=True)
holdout_pool = df.iloc[holdout_idx].reset_index(drop=True)

if len(train_pool) < max(SAMPLE_SIZES):
    raise ValueError("Training pool is smaller than the largest requested sample.")

# One fixed evaluation sample for all algorithms, feature sets, sizes and seeds.
eval_rng = np.random.default_rng(HOLDOUT_SEED)
eval_n = min(EVAL_SIZE, len(holdout_pool))
eval_positions = np.sort(eval_rng.choice(len(holdout_pool), size=eval_n, replace=False))
evaluation = holdout_pool.iloc[eval_positions].reset_index(drop=True)

split_summary = {
    "split_seed": HOLDOUT_SEED,
    "train_pool_rows": len(train_pool),
    "holdout_pool_rows": len(holdout_pool),
    "fixed_evaluation_rows": len(evaluation),
    "train_unique_invoices": int(train_pool["InvoiceNo"].nunique()),
    "holdout_unique_invoices": int(holdout_pool["InvoiceNo"].nunique()),
}
print(json.dumps(split_summary, indent=2))
with open(f"{OUTPUT_DIR}/split_summary.json", "w") as f:
    json.dump(split_summary, f, indent=2)


## 5. Define preprocessing and model factories

For each training subset and feature scenario, `log1p` is applied and the scaler is fitted on that training subset only. The same fitted scaler transforms the fixed evaluation sample for that model.

The two `n_init` settings are analyzed separately. In each comparison, KMeans and MiniBatchKMeans use the same `n_init` and random seed. Explicit MiniBatch parameters are recorded rather than relying on undocumented defaults.

In [ ]:
FEATURE_SCENARIOS = {
    "A_two_features": ["Quantity", "UnitPrice"],
    "B_three_features": ["Quantity", "UnitPrice", "TotalAmount"],
}

def transform_train_eval(train_frame, eval_frame, feature_cols):
    X_train_raw = train_frame[feature_cols].to_numpy(dtype=np.float64)
    X_eval_raw = eval_frame[feature_cols].to_numpy(dtype=np.float64)

    # Values are positive after cleaning, so log1p is defined.
    X_train_log = np.log1p(X_train_raw)
    X_eval_log = np.log1p(X_eval_raw)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_log)
    X_eval = scaler.transform(X_eval_log)

    return np.ascontiguousarray(X_train, dtype=np.float64), np.ascontiguousarray(X_eval, dtype=np.float64), scaler

def make_kmeans(seed, n_init):
    return KMeans(
        n_clusters=N_CLUSTERS,
        init="k-means++",
        n_init=n_init,
        max_iter=MAX_ITER,
        tol=TOL,
        algorithm="lloyd",
        random_state=seed
    )

def make_minibatch(seed, n_init):
    return MiniBatchKMeans(
        n_clusters=N_CLUSTERS,
        init="k-means++",
        n_init=n_init,
        max_iter=MAX_ITER,
        tol=TOL,
        batch_size=BATCH_SIZE,
        max_no_improvement=10,
        init_size=None,             # documented sklearn automatic behavior
        reassignment_ratio=0.01,
        random_state=seed
    )

PARAMETERS = {
    "n_clusters": N_CLUSTERS,
    "init": "k-means++",
    "max_iter": MAX_ITER,
    "tol": TOL,
    "n_init_values": N_INIT_VALUES,
    "kmeans_algorithm": "lloyd",
    "minibatch_batch_size": BATCH_SIZE,
    "minibatch_max_no_improvement": 10,
    "minibatch_init_size": None,
    "minibatch_reassignment_ratio": 0.01,
    "thread_limit": THREAD_LIMIT,
    "time_repeats": TIME_REPEATS,
    "sample_sizes": SAMPLE_SIZES,
    "seeds": SEED_LIST,
    "feature_scenarios": FEATURE_SCENARIOS,
}
with open(f"{OUTPUT_DIR}/experiment_parameters.json", "w") as f:
    json.dump(PARAMETERS, f, indent=2)
print(json.dumps(PARAMETERS, indent=2))


## 6. Run the experiment

For every seed, a single random permutation of the training pool is generated. Each larger sample contains all rows from the smaller sample for that same seed (nested sampling).

A warm-up fit is performed before recorded runs. Fit time is measured over `TIME_REPEATS` fresh model fits; the median and individual timings are stored. Evaluation metrics and ARI are computed from a deterministic fitted model using the same seed. Thread pools are limited to one thread for consistency within this run.

**Runtime note:** this is a substantial experiment (30 seeds × 6 sizes × 2 feature scenarios × 2 algorithms × 2 n_init settings). Run in Colab with a stable runtime. If interrupted, save intermediate results after each completed condition or rerun the notebook from the beginning.

In [ ]:
# Fixed row positions for the silhouette subset, shared by every fitted model.
sil_n = min(SILHOUETTE_SIZE, len(evaluation))
sil_rng = np.random.default_rng(HOLDOUT_SEED + 1)
sil_positions = np.sort(sil_rng.choice(len(evaluation), size=sil_n, replace=False))

def evaluate_partition(X_eval, labels, centers):
    # Internal metrics on the same fixed holdout evaluation rows.
    # Silhouette is computed on a fixed subset for computational feasibility.
    labels_sil = labels[sil_positions]
    X_sil = X_eval[sil_positions]

    if len(np.unique(labels)) < 2 or len(np.unique(labels)) >= len(labels):
        raise ValueError("Invalid number of predicted clusters for evaluation.")
    if len(np.unique(labels_sil)) < 2:
        sil = np.nan
    else:
        sil = silhouette_score(X_sil, labels_sil, metric="euclidean")

    db = davies_bouldin_score(X_eval, labels)
    ch = calinski_harabasz_score(X_eval, labels)
    # Evaluation inertia: sum of squared distances to assigned fitted centers,
    # normalized per evaluation observation for cross-size comparability.
    eval_inertia_per_sample = float(np.mean(np.sum((X_eval - centers[labels])**2, axis=1)))

    return {
        "silhouette_eval": float(sil),
        "davies_bouldin_eval": float(db),
        "calinski_harabasz_eval": float(ch),
        "eval_inertia_per_sample": eval_inertia_per_sample,
    }

def timed_fit(model_factory, X_train, seed, n_init):
    durations = []
    fitted = None
    for rep in range(TIME_REPEATS):
        model = model_factory(seed, n_init)
        gc.collect()
        start = time.perf_counter()
        with threadpool_limits(limits=THREAD_LIMIT):
            model.fit(X_train)
        durations.append(time.perf_counter() - start)
        if rep == 0:
            fitted = model
    return fitted, durations

# Warm-up: small, separate synthetic data; not included in recorded timings.
warm_rng = np.random.default_rng(999)
X_warm = warm_rng.normal(size=(max(500, N_CLUSTERS * 100), 2))
with threadpool_limits(limits=THREAD_LIMIT):
    make_kmeans(999, 1).fit(X_warm)
    make_minibatch(999, 1).fit(X_warm)
gc.collect()

results = []
start_all = time.time()

for feature_name, feature_cols in FEATURE_SCENARIOS.items():
    print("\nFEATURE SCENARIO:", feature_name, feature_cols)

    for seed in SEED_LIST:
        # One permutation per seed => nested subsets across sample sizes.
        rng = np.random.default_rng(seed)
        nested_order = rng.permutation(len(train_pool))

        for n in SAMPLE_SIZES:
            subset = train_pool.iloc[nested_order[:n]].reset_index(drop=True)
            X_train, X_eval, scaler = transform_train_eval(subset, evaluation, feature_cols)

            for n_init in N_INIT_VALUES:
                fitted_models = {}
                for algo_name, factory in [
                    ("KMeans", make_kmeans),
                    ("MiniBatchKMeans", make_minibatch),
                ]:
                    fitted, durations = timed_fit(factory, X_train, seed, n_init)
                    labels_eval = fitted.predict(X_eval)
                    metrics = evaluate_partition(X_eval, labels_eval, fitted.cluster_centers_)

                    row = {
                        "feature_scenario": feature_name,
                        "features": ",".join(feature_cols),
                        "seed": seed,
                        "sample_size": n,
                        "n_init": n_init,
                        "algorithm": algo_name,
                        "train_inertia": float(fitted.inertia_),
                        "train_inertia_per_sample": float(fitted.inertia_ / n),
                        "fit_time_median_seconds": float(np.median(durations)),
                        "fit_time_repeats_seconds": json.dumps([float(x) for x in durations]),
                        "n_iter": int(fitted.n_iter_),
                        "cluster_count_eval": int(len(np.unique(labels_eval))),
                        **metrics,
                    }
                    results.append(row)
                    fitted_models[algo_name] = labels_eval

                ari = adjusted_rand_score(
                    fitted_models["KMeans"],
                    fitted_models["MiniBatchKMeans"]
                )
                # Attach the paired ARI to both algorithm rows for convenient export.
                for row in results[-2:]:
                    row["ari_between_algorithms_eval"] = float(ari)

            # Persist checkpoint after every completed feature/seed/size condition.
            pd.DataFrame(results).to_csv(f"{OUTPUT_DIR}/experiment_results_checkpoint.csv", index=False)
            print(f"Completed {feature_name} | seed={seed} | n={n}")

print(f"\nFinished in {(time.time()-start_all)/60:.1f} minutes.")
results_df = pd.DataFrame(results)
results_df.to_csv(f"{OUTPUT_DIR}/experiment_results_raw.csv", index=False)
display(results_df.head())


## 7. Summarize results (mean, SD, paired differences)

The primary paired comparison is performed within the same feature scenario, seed, sample size and `n_init`. Report exact p-values and apply Holm correction separately within each clearly defined family of tests. Avoid describing correlated internal metrics as independent confirmations.

In [ ]:
metric_cols = [
    "silhouette_eval", "davies_bouldin_eval", "calinski_harabasz_eval",
    "eval_inertia_per_sample", "fit_time_median_seconds",
    "train_inertia", "train_inertia_per_sample"
]

summary = (
    results_df
    .groupby(["feature_scenario", "sample_size", "n_init", "algorithm"])[metric_cols]
    .agg(["mean", "std"])
    .reset_index()
)
summary.to_csv(f"{OUTPUT_DIR}/summary_mean_sd.csv", index=False)
display(summary)


In [ ]:
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# Paired differences: MiniBatchKMeans minus KMeans.
# For Silhouette/CH higher is better; for DB/inertia/time lower is better.
paired_metrics = [
    "silhouette_eval", "davies_bouldin_eval",
    "calinski_harabasz_eval", "eval_inertia_per_sample",
    "fit_time_median_seconds"
]

paired_rows = []
group_cols = ["feature_scenario", "sample_size", "n_init"]

for keys, group in results_df.groupby(group_cols):
    wide = group.pivot(index="seed", columns="algorithm", values=paired_metrics)
    for metric in paired_metrics:
        a = wide[(metric, "MiniBatchKMeans")].dropna()
        b = wide[(metric, "KMeans")].dropna()
        common = a.index.intersection(b.index)
        a, b = a.loc[common], b.loc[common]
        diff = a - b
        if len(diff) == 0 or np.allclose(diff.to_numpy(), 0):
            stat, p = np.nan, 1.0
        else:
            stat, p = wilcoxon(a, b, alternative="two-sided", zero_method="wilcox", method="auto")
        # Matched-pairs rank-biserial correlation effect size.
        nonzero = diff[diff != 0]
        if len(nonzero):
            ranks = nonzero.abs().rank(method="average")
            w_pos = ranks[nonzero > 0].sum()
            w_neg = ranks[nonzero < 0].sum()
            rbc = float((w_pos - w_neg) / (w_pos + w_neg)) if (w_pos + w_neg) else 0.0
        else:
            rbc = 0.0

        paired_rows.append({
            "feature_scenario": keys[0],
            "sample_size": keys[1],
            "n_init": keys[2],
            "metric": metric,
            "n_pairs": len(diff),
            "mean_paired_difference_MB_minus_KM": float(diff.mean()),
            "median_paired_difference_MB_minus_KM": float(diff.median()),
            "wilcoxon_statistic": float(stat) if np.isfinite(stat) else np.nan,
            "p_value_exact_as_returned": float(p),
            "rank_biserial_correlation": rbc,
        })

paired_df = pd.DataFrame(paired_rows)
# Holm correction within each feature scenario × n_init family across six sample sizes × metrics.
paired_df["p_holm"] = np.nan
for _, idx in paired_df.groupby(["feature_scenario", "n_init"]).groups.items():
    pvals = paired_df.loc[idx, "p_value_exact_as_returned"].fillna(1.0).to_numpy()
    paired_df.loc[idx, "p_holm"] = multipletests(pvals, method="holm")[1]

paired_df.to_csv(f"{OUTPUT_DIR}/paired_tests_wilcoxon_holm.csv", index=False)
display(paired_df)


## 8. Export reproducibility metadata and download outputs

In [ ]:
metadata = {
    "experiment_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "threadpool_info": threadpool_info(),
    "parameters": PARAMETERS,
    "split": split_summary,
    "cleaning_counts": cleaning_summary.to_dict(orient="records"),
    "dataset_sha256": hashlib.sha256(open(DATA_PATH, "rb").read()).hexdigest(),
}
with open(f"{OUTPUT_DIR}/run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

# Package all outputs into one ZIP file.
import shutil
zip_path = shutil.make_archive("/content/online_retail_experiment_outputs", "zip", OUTPUT_DIR)
print("Created:", zip_path)

from google.colab import files
files.download(zip_path)


## 9. Interpretation and reporting checklist

Before updating the manuscript:

1. Check whether every condition completed: 2 feature scenarios × 30 seeds × 6 sample sizes × 2 n_init values × 2 algorithms.
2. Inspect cluster counts. A condition with fewer than 5 predicted clusters must be investigated; its internal metrics may be invalid or incomparable.
3. Report mean ± SD across 30 seeds, and exact paired-test p-values with Holm-adjusted p-values.
4. Report the effect-size definition (matched-pairs rank-biserial correlation) explicitly.
5. Treat Silhouette, Davies–Bouldin and Calinski–Harabasz as internal validation metrics, not business usefulness.
6. Do not claim MiniBatch is faster or better unless the measured results support it.
7. Do not report memory differences from this notebook: memory profiling is intentionally excluded from the primary experiment because noisy process-level memory measurements in hosted notebooks can be misleading. If memory is a required outcome, design a separate, documented process-isolated memory experiment.
8. State the exact Colab runtime hardware shown in `run_metadata.json`; Colab hardware may vary across sessions.
9. Upload the notebook and output archive to a public repository only after removing any private information and confirming the dataset's redistribution terms.
10. Replace the old article's results and conclusions with the new outputs; do not mix old and new experimental results.
